# Phase 9 — Ablation study sur l’architecture Keras

## Objectif

Faire une **ablation study** sur MNIST pour comprendre l’impact de :
- la profondeur du réseau ;
- la largeur des couches cachées.

On garde :
- le même dataset : MNIST ;
- le même optimizer : Adam (`lr=1e-3`) ;
- la même activation : ReLU ;
- le même nombre d’epochs : 15.

On fait varier :
- `depth = [1, 2, 3]`
- `width = [8, 64, 256]`

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0

depths = [1, 2, 3]
widths = [8, 64, 256]
results = []

print("Train :", X_train.shape, y_train.shape)
print("Test  :", X_test.shape, y_test.shape)

Train : (60000, 784) (60000,)
Test  : (10000, 784) (10000,)


## 1. Boucle d’ablation

Pour chaque combinaison `(depth, width)` :
1. construire le modèle ;
2. compter les paramètres ;
3. entraîner ;
4. mesurer le temps ;
5. évaluer sur le test set ;
6. repérer la première epoch où `val_accuracy > 0.95`.

In [3]:
for depth in depths:
    for width in widths:
        tf.random.set_seed(42)

        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(784,)))

        for _ in range(depth):
            model.add(keras.layers.Dense(width, activation="relu"))

        model.add(keras.layers.Dense(10, activation="softmax"))

        n_params = model.count_params()

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        start = time.time()

        history = model.fit(
            X_train,
            y_train,
            epochs=15,
            batch_size=64,
            validation_split=0.1,
            verbose=0
        )

        train_time = time.time() - start
        test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

        val_loss_final = history.history["val_loss"][-1]
        val_accs = history.history["val_accuracy"]

        epoch_95 = next(
            (i + 1 for i, acc in enumerate(val_accs) if acc > 0.95),
            "N/A"
        )

        results.append({
            "depth": depth,
            "width": width,
            "n_params": n_params,
            "test_accuracy": test_acc,
            "val_loss_final": val_loss_final,
            "epoch_95": epoch_95,
            "train_time_s": train_time
        })

In [4]:
print("\n=== ABLATION STUDY : PROFONDEUR x LARGEUR ===")
print(f"{'Depth':6s} | {'Width':6s} | {'Params':10s} | {'Test acc':10s} | {'Val loss':10s} | {'Epoch >95%':10s} | {'Temps (s)':10s}")
print("-" * 95)

for r in results:
    print(
        f"{r['depth']:6d} | "
        f"{r['width']:6d} | "
        f"{r['n_params']:10,d} | "
        f"{r['test_accuracy']:.4f} | "
        f"{r['val_loss_final']:.4f} | "
        f"{str(r['epoch_95']):10s} | "
        f"{r['train_time_s']:.0f}"
    )


=== ABLATION STUDY : PROFONDEUR x LARGEUR ===
Depth  | Width  | Params     | Test acc   | Val loss   | Epoch >95% | Temps (s) 
-----------------------------------------------------------------------------------------------
     1 |      8 |      6,370 | 0.9215 | 0.2365 | N/A        | 27
     1 |     64 |     50,890 | 0.9713 | 0.1013 | 1          | 38
     1 |    256 |    203,530 | 0.9749 | 0.1038 | 1          | 81
     2 |      8 |      6,442 | 0.9276 | 0.2157 | N/A        | 78
     2 |     64 |     55,050 | 0.9706 | 0.1094 | 1          | 80
     2 |    256 |    269,322 | 0.9786 | 0.1160 | 1          | 100
     3 |      8 |      6,514 | 0.9223 | 0.2280 | N/A        | 36
     3 |     64 |     59,210 | 0.9691 | 0.1254 | 1          | 51
     3 |    256 |    335,114 | 0.9758 | 0.0894 | 1          | 94


In [5]:
acc_grid = np.array([
    [r["test_accuracy"] for r in results if r["depth"] == d]
    for d in depths
])

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(acc_grid, cmap="RdYlGn", vmin=0.85, vmax=0.99)

ax.set_xticks(range(len(widths)))
ax.set_xticklabels(widths)
ax.set_yticks(range(len(depths)))
ax.set_yticklabels(depths)

ax.set_xlabel("Largeur (neurones par couche)")
ax.set_ylabel("Profondeur (couches cachées)")
ax.set_title("Test accuracy selon l'architecture (MNIST)")

for i in range(len(depths)):
    for j in range(len(widths)):
        ax.text(j, i, f"{acc_grid[i, j]:.3f}", ha="center", va="center", fontsize=9)

plt.colorbar(im)
plt.tight_layout()
plt.savefig("phase9_ablation_heatmap.png", dpi=100, bbox_inches="tight")
plt.show()

print("Heatmap sauvegardée : phase9_ablation_heatmap.png")

Heatmap sauvegardée : phase9_ablation_heatmap.png


C:\Users\serge\AppData\Local\Temp\ipykernel_31448\4223774601.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Analyse attendue

Le support indique que MNIST est un problème simple :
- au-delà de `depth=2` / `width=64`, le gain devient marginal ;
- il faut identifier le point de rendements décroissants ;
- il faut repérer la configuration la plus légère qui atteint > 97% de test accuracy.

Question à commenter :
- pourquoi ne pas toujours prendre le plus gros modèle ?